In [2]:
import math
import csv


def load_csv(filename):
    with open(filename, "r") as f:
        lines = csv.reader(f)
        dataset = list(lines)
    headers = dataset.pop(0)
    return dataset, headers


class Node:
    def __init__(self, attribute):
        self.attribute = attribute
        self.children = []
        self.answer = ""


def subtables(data, col, delete):
    dic = {}
    coldata = [row[col] for row in data]
    attr = list(set(coldata))

    counts = [0] * len(attr)
    r = len(data)
    c = len(data[0])

    for x in range(len(attr)):
        for y in range(r):
            if data[y][col] == attr[x]:
                counts[x] += 1

    for x in range(len(attr)):
        dic[attr[x]] = [[0 for _ in range(c)] for _ in range(counts[x])]

        pos = 0
        for y in range(r):
            if data[y][col] == attr[x]:
                if delete:
                    row = data[y][:col] + data[y][col+1:]
                else:
                    row = data[y][:]
                dic[attr[x]][pos] = row
                pos += 1

    return attr, dic


def entropy(S):
    attr = list(set(S))
    if len(attr) == 1:
        return 0

    counts = [0] * len(attr)
    for i in range(len(attr)):
        counts[i] = sum(1 for x in S if x == attr[i]) / float(len(S))

    sums = 0
    for cnt in counts:
        if cnt > 0:
            sums += -cnt * math.log(cnt, 2)

    return sums


def compute_gain(data, col):
    attr, dic = subtables(data, col, delete=False)

    total_size = len(data)
    total_entropy = entropy([row[-1] for row in data])

    for x in range(len(attr)):
        ratio = len(dic[attr[x]]) / float(total_size)
        ent = entropy([row[-1] for row in dic[attr[x]]])
        total_entropy -= ratio * ent

    return total_entropy


def build_tree(data, features):
    lastcol = [row[-1] for row in data]

    if len(set(lastcol)) == 1:
        node = Node("")
        node.answer = lastcol[0]
        return node

    n = len(data[0]) - 1
    gains = [compute_gain(data, col) for col in range(n)]

    split = gains.index(max(gains))
    node = Node(features[split])

    fea = features[:split] + features[split+1:]
    attr, dic = subtables(data, split, delete=True)

    for x in range(len(attr)):
        child = build_tree(dic[attr[x]], fea)
        node.children.append((attr[x], child))

    return node


def print_tree(node, level):
    if node.answer != "":
        print("\t" * level, node.answer)
        return

    print("\t" * level, node.attribute)
    for value, n in node.children:
        print("\t" * (level + 1), value)
        print_tree(n, level + 2)


def classify(node, x_test, features):
    if node.answer != "":
        print(node.answer)
        return

    pos = features.index(node.attribute)
    for value, n in node.children:
        if x_test[pos] == value:
            classify(n, x_test, features)


# -------- Main Program --------
dataset, features = load_csv("football_data.csv")
node1 = build_tree(dataset, features)

print("The decision tree for the dataset using ID3 algorithm is:")
print_tree(node1, 0)

testdata, features = load_csv("football_data.csv")

for xtest in testdata:
    print("The test instance:", xtest)
    print("The label for test instance:", end="\t")
    classify(node1, xtest, features)

The decision tree for the dataset using ID3 algorithm is:
 Day
	 Day 13
		 Yes
	 Day 2
		 No
	 Day 15
		 No
	 Day 11
		 Yes
	 Day 14
		 Yes
	 Day 8
		 No
	 Day 12
		 Yes
	 Day 6
		 No
	 Day 1
		 No
	 Day 10
		 Yes
	 Day 3
		 Yes
	 Day 4
		 Yes
	 Day 7
		 Yes
	 Day 5
		 Yes
	 Day 9
		 Yes
The test instance: ['Day 1', 'Sunny', 'Hot', 'High', 'Weak', 'No']
The label for test instance:	No
The test instance: ['Day 2', 'Sunny', 'Hot', 'High', 'Strong', 'No']
The label for test instance:	No
The test instance: ['Day 3', 'Cloudy', 'Hot', 'High', 'Weak', 'Yes']
The label for test instance:	Yes
The test instance: ['Day 4', 'Rainy', 'Mild', 'High', 'Weak', 'Yes']
The label for test instance:	Yes
The test instance: ['Day 5', 'Rainy', 'Cool', 'Normal', 'Weak', 'Yes']
The label for test instance:	Yes
The test instance: ['Day 6', 'Rainy', 'Cool', 'Normal', 'Strong', 'No']
The label for test instance:	No
The test instance: ['Day 7', 'Cloudy', 'Cool', 'Normal', 'Strong', 'Yes']
The label for test instan